# Copay Accumulator Program Audit

When a member uses a manufacturer copay card at the pharmacy, the card pays
some or all of the member's copay. The question is whether that manufacturer
payment gets credited toward the member's deductible and out-of-pocket maximum.

On a TRADITIONAL plan, it should. The member's deductible goes down regardless
of who paid. On an ACCUMULATOR plan, it should not. The whole point of the
accumulator program is to exclude manufacturer assistance from the member's
running totals, so the member continues to owe cost-sharing out of pocket
even while the copay card covers their fill-by-fill copay.

This notebook audits pharmacy claims to find cases where the PBM incorrectly
credited manufacturer copay payments toward accumulator totals on ACCUMULATOR
plans. It also checks for a separate but related issue: plans that are using
an ACCUMULATOR program in a state where the law prohibits it.

In [0]:
%run ./NB_00_pipeline_utilities

# NB00 - Pipeline Utilities
**Eligibility & Accumulator**

In production, a reconciliation pipeline has dozens of checks spread across multiple notebooks.
If one those check fails, say the TPA eligibility table got dropped by an upstream job, you don't
want the entire pipeline to crash and leave you with nothing. You still want the accumulator
reconciliation, the copay audit, and the rollover checks to run and produce findings.

You also need to know exactly what ran, what passed, and what eventually broke.

This notebook provides shared utilities that every downstream notebook imports via
`%run ./NB_00_pipeline_utilities`. It handles three things:

| Utility | What it does |
|---------|-------------|
| Structured logging | JSON audit entries to `pbm_sim.pipeline_audit_log`, queryable by run, notebook, section |
| Error handling decorator | Wraps each check in try/except so one failure doesn't kill the entire pipeline |
| Table validation helpers | Pre-flight checks before a notebook starts its logic (do the table exists? does it has rows? does it have right columns?) |

These functions are used in NB01 through NB07.

## Structured Logging

Every pipeline run gets a unique `run_id` based on the UTC timestamp when the run started.
It ties together all the log entries from a single execution across notebooks.
If the orchestrator kicks off NB01 through NB07 sequentially, they all share the same run_id.

Log entries go to two places:
1. **stdout** as `[PIPELINE_LOG] {...}` - captured by Databricks job logs, useful for real-time monitoring
2. **Delta table** `pbm_sim.pipeline_audit_log` - the permanent audit trail you can query later

Each entry records the notebook, section, outcome (SUCCESS/WARNING/ERROR/SKIPPED),
row count, timing, and a free-text detail field. The detail field is truncated at 2,000
characters to prevent a full stack trace from bloating the table.

## Error Handling Decorator

This is the core safety mechanism. In a reconciliation pipeline, each notebook runs
multiple independent checks. Take NB03 (accumulator reconciliation) - it checks for
deductible overcharges AND OOPM overcharges. If the deductible check throws an exception
(maybe a column was renamed upstream), you still want the OOPM check to run and produce
its findings. Without this decorator, an unhandled exception in the deductible check
would kill the notebook, and you'd lose the OOPM results too.

The decorator does three things on failure:
1. Catches the exception and logs it as ERROR with the full traceback
2. Returns an empty DataFrame with the expected schema so downstream unions don't break
3. Lets the notebook continue to the next section

On success, it logs the section as SUCCESS with the row count and wall-clock time.

**Usage in a downstream notebook:**
```python
@pipeline_section("NB_03", "deductible_overcharge", finding_schema=convergence_schema)
def run_deductible_check():
    # ... compute df ...
    return df

df_result = run_deductible_check()
# If the check fails, df_result is an empty DataFrame with the right columns
# so the union at the end of the notebook still works
```

## Table Validation Helpers (Pre-flight Checks)

Before a notebook starts its actual logic, it should verify that the tables it depends on
exist and have data. Without this, a missing table produces a cryptic *AnalysisException*
50 cells into the notebook, and you spend 10 minutes figuring out that the problem was
simply that NB00 didn't run.

These are the pre-flight checks:
- **The Table exists?** Catches the case where the upstream notebook didn't run or the table got dropped
- **Are the table empty?** The table exists but has zero rows - maybe the data generator ran but the filter
  produced nothing. This is different from "table doesn't exist" and needs a different response
- **Are the required columns present?** Schema drift - a column was renamed or removed upstream. Better to catch
  this up front with a clear message than to get a "Column 'xyz' not found" error mid-analysis

## Pipeline Metadata

Convenience function for embedding run metadata into output tables.
When NB07 unions all findings, each row can carry the run_id so you can
trace any finding back to the exact pipeline execution that produced it.

Pipeline utilities loaded. Run ID: RUN_20261017_192721


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE pbm_sim")

NB = "NB_04"

# Pre-flight - we need pharmacy claims and plan design.
# The plan_design table carries the accumulator_program_type that tells us
# whether manufacturer copay should count or not.
for tbl in ["pbm_sim.pharmacy_claims", "pbm_sim.plan_design"]:
    validate_table_exists(tbl, NB)

df_claims = spark.table("pbm_sim.pharmacy_claims")
df_plan = spark.table("pbm_sim.plan_design")

claims_count = df_claims.count()
plan_count = df_plan.count()

log_entry(NB, "load_source_tables", "SUCCESS",
          detail=f"Pharmacy claims: {claims_count}, Plan designs: {plan_count}")

print(f"Pharmacy claims loaded: {claims_count}")
print(f"Plan designs loaded:    {plan_count}")

[PIPELINE_LOG] {"run_id": "RUN_20261017_192721", "timestamp": "2026-10-17T19:27:24.211827+00:00", "notebook": "NB_04", "section": "load_source_tables", "status": "SUCCESS", "row_count": 0, "detail": "Pharmacy claims: 400040, Plan designs: 40", "duration_sec": 0.0}
Pharmacy claims loaded: 400040
Plan designs loaded:    40


## 1. Plan Configuration Review

First, let's see which plans use which program type. This is the reference
we'll audit against. Any claim on an ACCUMULATOR plan where the manufacturer
copay was counted toward the deductible is an error.

In [0]:
display(
    df_plan
    .select(
        "plan_id", "plan_name", "accumulator_program_type",
        "funding_model", "state_domicile"
    )
    .orderBy("plan_id")
)

plan_id,plan_name,accumulator_program_type,funding_model,state_domicile
PLN001,Alpha Corp,TRADITIONAL,SELF_FUNDED,CT
PLN002,Beta Industries,ACCUMULATOR,SELF_FUNDED,NJ
PLN003,Gamma Health Trust,TRADITIONAL,FULLY_INSURED,NY
PLN004,Delta Logistics,ACCUMULATOR,SELF_FUNDED,PA
PLN005,Epsilon Teachers Fund,ACCUMULATOR,FULLY_INSURED,AZ
PLN006,Zeta Manufacturing,MAXIMIZER,SELF_FUNDED,OH
PLN007,Eta Municipal Workers,TRADITIONAL,FULLY_INSURED,GA
PLN008,Theta Retail Group,TRADITIONAL,SELF_FUNDED,IN
PLN009,Iota Financial Services,TRADITIONAL,SELF_FUNDED,OH
PLN010,Kappa Tech Solutions,ACCUMULATOR,SELF_FUNDED,CA


## 2. Manufacturer Copay Card Usage

Before looking for errors, it helps to understand the baseline. How many
claims involve a copay card at all? Which plans and drugs are they
concentrated in? This context matters because copay card usage is
heavily skewed toward specialty drugs, and specialty drugs are exactly
where ACCUMULATOR programs have the biggest financial impact.

In [0]:
df_coupon_claims = df_claims.filter(F.col("manufacturer_copay_amount") > 0)
total_claims = df_claims.count()
coupon_count = df_coupon_claims.count()

print(f"Total pharmacy claims:            {total_claims}")
print(f"Claims with manufacturer copay:   {coupon_count}")
print(f"Copay card utilization rate:       {coupon_count / total_claims * 100:.1f}%")

log_entry(NB, "copay_card_baseline", "SUCCESS",
          row_count=coupon_count,
          detail=f"{coupon_count}/{total_claims} claims used copay cards "
                 f"({coupon_count / total_claims * 100:.1f}%)")

Total pharmacy claims:            400040
Claims with manufacturer copay:   32163
Copay card utilization rate:       8.0%
[PIPELINE_LOG] {"run_id": "RUN_20261017_192721", "timestamp": "2026-10-17T19:27:32.875310+00:00", "notebook": "NB_04", "section": "copay_card_baseline", "status": "SUCCESS", "row_count": 32163, "detail": "32163/400040 claims used copay cards (8.0%)", "duration_sec": 0.0}


In [0]:
# Copay card usage by drug.
# Specialty drugs dominate this list because that's where the drug costs
# are high enough for manufacturers to offer copay assistance programs.
print("Copay card claims by drug (top drugs):")
display(
    df_coupon_claims
    .groupBy("drug_name")
    .agg(
        F.count("*").alias("claims"),
        F.round(F.sum("manufacturer_copay_amount"), 2).alias("total_mfr_paid"),
        F.round(F.avg("manufacturer_copay_amount"), 2).alias("avg_mfr_paid"),
    )
    .orderBy(F.col("claims").desc())
)

Copay card claims by drug (top drugs):


drug_name,claims,total_mfr_paid,avg_mfr_paid
Humira 40mg Pen,4102,4219293.94,1028.59
Enbrel 50mg,4066,3777451.11,929.03
Cosentyx 150mg,4048,3373630.44,833.41
Keytruda 100mg,4033,6005373.78,1489.06
Dupixent 300mg,4032,2119546.5,525.68
Ozempic 1mg,3999,711257.3,177.86
Trulicity 1.5mg,3949,567800.82,143.78
Stelara 45mg,3934,4400326.89,1118.54


## 3. Copay Accumulator Errors (ACCUMULATOR Plans)

The core audit. We join claims with plan design and look for the combination
that should never occur: a claim on an ACCUMULATOR plan where the manufacturer
copay was counted toward the deductible (flag = 'Y').

Each of these is a processing error. The member's deductible balance was
reduced by the manufacturer's payment when it should not have been. This
means the member will hit their deductible sooner than they should, and the
plan starts paying 100% earlier than the benefit design intended.

In [0]:
df_audit = (
    df_coupon_claims
    .join(df_plan, on="plan_id")
    .filter(
        (F.col("accumulator_program_type") == "ACCUMULATOR") &
        (F.col("manufacturer_copay_counted_toward_deductible") == "Y")
    )
)

error_count = df_audit.count()
error_dollars = df_audit.agg(F.sum("manufacturer_copay_amount")).collect()[0][0] or 0

print(f"ACCUMULATOR plan copay errors found: {error_count}")
print(f"Total manufacturer dollars incorrectly credited: ${error_dollars:,.2f}")

log_entry(NB, "accumulator_errors", "WARNING" if error_count > 0 else "SUCCESS",
          row_count=error_count,
          detail=f"{error_count} claims with mfr copay incorrectly credited, "
                 f"${error_dollars:,.2f}")

ACCUMULATOR plan copay errors found: 60
Total manufacturer dollars incorrectly credited: $51,082.96
[PIPELINE_LOG] {"run_id": "RUN_20261017_192721", "timestamp": "2026-10-17T19:27:37.792702+00:00", "notebook": "NB_04", "section": "accumulator_errors", "status": "WARNING", "row_count": 60, "detail": "60 claims with mfr copay incorrectly credited, $51,082.96", "duration_sec": 0.0}


In [0]:
if error_count > 0:
    print("Error detail (each row is a claim that was incorrectly processed):")
    display(
        df_audit
        .select(
            "claim_id", "member_id", "plan_id", "plan_name",
            "service_date", "drug_name", "drug_cost",
            "member_copay", "manufacturer_copay_amount",
            "manufacturer_copay_counted_toward_deductible",
            "funding_model", "state_domicile"
        )
        .orderBy("plan_id", "service_date")
    )

Error detail (each row is a claim that was incorrectly processed):


claim_id,member_id,plan_id,plan_name,service_date,drug_name,drug_cost,member_copay,manufacturer_copay_amount,manufacturer_copay_counted_toward_deductible,funding_model,state_domicile
CLM9000010252,MBR1003645,PLN002,Beta Industries,2026-01-05,Enbrel 50mg,6508.82,1381.55,1177.7,Y,SELF_FUNDED,NJ
CLM9000008186,MBR1002765,PLN002,Beta Industries,2026-02-13,Keytruda 100mg,10314.7,1071.21,920.62,Y,SELF_FUNDED,NJ
CLM9000007709,MBR1003331,PLN002,Beta Industries,2026-03-05,Keytruda 100mg,11150.26,2961.68,2333.58,Y,SELF_FUNDED,NJ
CLM9000014009,MBR1004187,PLN002,Beta Industries,2026-03-28,Stelara 45mg,8705.74,2041.42,1348.4,Y,SELF_FUNDED,NJ
CLM9000017203,MBR1003661,PLN002,Beta Industries,2026-04-19,Dupixent 300mg,3905.52,1122.18,943.45,Y,SELF_FUNDED,NJ
CLM9000003680,MBR1002585,PLN002,Beta Industries,2026-04-22,Humira 40mg Pen,8146.2,2045.07,1024.74,Y,SELF_FUNDED,NJ
CLM9000003175,MBR1002465,PLN002,Beta Industries,2026-06-28,Keytruda 100mg,10341.68,1826.04,1565.93,Y,SELF_FUNDED,NJ
CLM9000013290,MBR1002496,PLN002,Beta Industries,2026-08-10,Trulicity 1.5mg,1099.59,224.38,215.02,Y,SELF_FUNDED,NJ
CLM9000024105,MBR1004179,PLN002,Beta Industries,2026-08-14,Keytruda 100mg,8616.04,2347.47,2200.91,Y,SELF_FUNDED,NJ
CLM9000009662,MBR1009093,PLN004,Delta Logistics,2026-01-12,Enbrel 50mg,6585.14,1300.02,1201.46,Y,SELF_FUNDED,PA


## 3b. MAXIMIZER Program Audit

MAXIMIZER plans require a more nuanced check than ACCUMULATOR plans.
The manufacturer copay card has a finite dollar value per benefit year.
While the card has funds, manufacturer payments should NOT count toward
accumulators (same rule as ACCUMULATOR). Once the card is exhausted
and the member starts paying out of pocket, those member payments
SHOULD count normally.

The audit identifies:
1. Claims where the copay card was active and manufacturer dollars were
   incorrectly credited (same error as ACCUMULATOR)
2. Copay card exhaustion events (the transition point per member per drug)
3. Post-exhaustion claims where member out-of-pocket was NOT credited
   (the inverse error - once the card is gone, the member's own money
   should count)

In [0]:
# Identify MAXIMIZER plans and their claims with copay card activity
df_maximizer_claims = (
    df_claims
    .join(df_plan, on="plan_id")
    .filter(F.col("accumulator_program_type") == "MAXIMIZER")
)

maximizer_total = df_maximizer_claims.count()
maximizer_coupon_claims = df_maximizer_claims.filter(F.col("manufacturer_copay_amount") > 0)
maximizer_coupon_count = maximizer_coupon_claims.count()

print(f"Total claims on MAXIMIZER plans: {maximizer_total}")
print(f"Of those, claims with copay card activity: {maximizer_coupon_count}")

Total claims on MAXIMIZER plans: 47507
Of those, claims with copay card activity: 3848


In [0]:
# Track cumulative manufacturer copay spend per member per drug per benefit year.
# This simulates copay card exhaustion detection. In production, the PBM would
# have the actual copay card max from the manufacturer's program terms.
# Here we use a proxy: a running total that lets us identify the spending pattern
# and flag the point where manufacturer payments drop to zero (exhaustion signal).

if maximizer_coupon_count > 0:
    w_cumulative = (
        Window
        .partitionBy("member_id", "drug_name", "benefit_year")
        .orderBy("service_date")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )

    df_maximizer_timeline = (
        df_maximizer_claims
        .withColumn(
            "cumulative_mfr_copay",
            F.round(F.sum("manufacturer_copay_amount").over(w_cumulative), 2)
        )
        .withColumn(
            "prior_mfr_copay",
            F.lag("manufacturer_copay_amount", 1, 0).over(
                Window.partitionBy("member_id", "drug_name", "benefit_year")
                .orderBy("service_date")
            )
        )
        .withColumn(
            "copay_card_phase",
            F.when(
                F.col("manufacturer_copay_amount") > 0, "PRE_EXHAUSTION"
            ).when(
                (F.col("manufacturer_copay_amount") == 0) & (F.col("cumulative_mfr_copay") > 0),
                "POST_EXHAUSTION"
            ).otherwise("NO_COPAY_CARD")
        )
    )

    # Error type 1: pre-exhaustion claims where manufacturer copay was credited.
    # Same error as ACCUMULATOR plans - the card is still active, so the
    # manufacturer's money should not be counting toward the member's accumulators.
    df_max_pre_errors = df_maximizer_timeline.filter(
        (F.col("copay_card_phase") == "PRE_EXHAUSTION") &
        (F.col("manufacturer_copay_counted_toward_deductible") == "Y")
    )

    max_pre_error_count = df_max_pre_errors.count()
    max_pre_error_dollars = df_max_pre_errors.agg(
        F.sum("manufacturer_copay_amount")
    ).collect()[0][0] or 0

    print(f"\nMAXIMIZER pre-exhaustion errors (mfr copay credited when it shouldn't be):")
    print(f"  Error claims: {max_pre_error_count}")
    print(f"  Dollars incorrectly credited: ${max_pre_error_dollars:,.2f}")

    # Error type 2: post-exhaustion claims where member's own payment was NOT credited.
    # This is the more harmful error - the copay card is gone, the member is paying
    # from their own pocket, but the system still isn't counting it toward their
    # deductible. The member is getting double-penalized.
    df_max_post_errors = df_maximizer_timeline.filter(
        (F.col("copay_card_phase") == "POST_EXHAUSTION") &
        (F.col("member_deductible_applied") == 0) &
        (F.col("member_copay") > 0)
    )

    max_post_error_count = df_max_post_errors.count()
    max_post_error_dollars = df_max_post_errors.agg(
        F.sum("member_copay")
    ).collect()[0][0] or 0

    print(f"\nMAXIMIZER post-exhaustion errors (member OOP not credited after card exhaustion):")
    print(f"  Error claims: {max_post_error_count}")
    print(f"  Member dollars not credited: ${max_post_error_dollars:,.2f}")

    # Exhaustion events summary - when did each member's copay card run out?
    df_exhaustion_events = (
        df_maximizer_timeline
        .filter(F.col("copay_card_phase") == "POST_EXHAUSTION")
        .groupBy("member_id", "drug_name", "plan_id", "benefit_year")
        .agg(
            F.min("service_date").alias("exhaustion_date"),
            F.max("cumulative_mfr_copay").alias("total_mfr_paid_before_exhaustion"),
            F.count("*").alias("post_exhaustion_claims"),
        )
        .orderBy("plan_id", "member_id")
    )

    exhaustion_count = df_exhaustion_events.count()
    print(f"\nCopay card exhaustion events detected: {exhaustion_count}")
    if exhaustion_count > 0:
        display(df_exhaustion_events)

    log_entry(NB, "maximizer_errors", "WARNING" if (max_pre_error_count + max_post_error_count) > 0 else "SUCCESS",
              row_count=max_pre_error_count + max_post_error_count,
              detail=f"Pre-exhaustion: {max_pre_error_count} errors (${max_pre_error_dollars:,.2f}), "
                     f"Post-exhaustion: {max_post_error_count} errors (${max_post_error_dollars:,.2f})")
else:
    max_pre_error_count = 0
    max_pre_error_dollars = 0
    max_post_error_count = 0
    max_post_error_dollars = 0
    print("No copay card activity on MAXIMIZER plans to audit.")
    log_entry(NB, "maximizer_errors", "SUCCESS",
              detail="No copay card activity on MAXIMIZER plans")


MAXIMIZER pre-exhaustion errors (mfr copay credited when it shouldn't be):
  Error claims: 3848
  Dollars incorrectly credited: $3,000,498.97

MAXIMIZER post-exhaustion errors (member OOP not credited after card exhaustion):
  Error claims: 0
  Member dollars not credited: $0.00

Copay card exhaustion events detected: 239


member_id,drug_name,plan_id,benefit_year,exhaustion_date,total_mfr_paid_before_exhaustion,post_exhaustion_claims
MBR1011839,Dupixent 300mg,PLN006,2026,2026-08-25,497.11,1
MBR1011840,Stelara 45mg,PLN006,2026,2026-09-24,2302.85,1
MBR1011845,Humira 40mg Pen,PLN006,2026,2026-07-25,864.86,1
MBR1011866,Enbrel 50mg,PLN006,2026,2026-09-16,749.36,1
MBR1011885,Trulicity 1.5mg,PLN006,2026,2026-04-20,236.79,1
MBR1011888,Enbrel 50mg,PLN006,2026,2026-09-02,873.13,1
MBR1011909,Dupixent 300mg,PLN006,2026,2026-03-13,391.06,1
MBR1011918,Ozempic 1mg,PLN006,2026,2026-09-13,99.98,1
MBR1011955,Humira 40mg Pen,PLN006,2026,2026-07-17,1001.03,1
MBR1012000,Enbrel 50mg,PLN006,2026,2026-04-02,1403.49,1


[PIPELINE_LOG] {"run_id": "RUN_20261017_192721", "timestamp": "2026-10-17T19:27:50.552101+00:00", "notebook": "NB_04", "section": "maximizer_errors", "status": "WARNING", "row_count": 3848, "detail": "Pre-exhaustion: 3848 errors ($3,000,498.97), Post-exhaustion: 0 errors ($0.00)", "duration_sec": 0.0}


## 4. Errors by Plan

Are the errors concentrated in one plan or spread across all ACCUMULATOR
plans? Concentration points to a plan-specific configuration issue.
Even distribution suggests a systemic adjudication logic bug.

In [0]:
if error_count > 0:
    print("Copay accumulator errors by plan:")
    display(
        df_audit
        .groupBy("plan_id", "plan_name", "funding_model", "state_domicile")
        .agg(
            F.count("*").alias("error_count"),
            F.round(F.sum("manufacturer_copay_amount"), 2).alias("total_mfr_incorrectly_credited"),
            F.countDistinct("member_id").alias("affected_members"),
        )
        .orderBy(F.col("error_count").desc())
    )

Copay accumulator errors by plan:


plan_id,plan_name,funding_model,state_domicile,error_count,total_mfr_incorrectly_credited,affected_members
PLN004,Delta Logistics,SELF_FUNDED,PA,12,9282.29,12
PLN002,Beta Industries,SELF_FUNDED,NJ,9,11730.35,9
PLN005,Epsilon Teachers Fund,FULLY_INSURED,AZ,8,4268.83,8
PLN010,Kappa Tech Solutions,SELF_FUNDED,CA,7,4467.31,7
PLN029,Evergreen School District,FULLY_INSURED,TN,5,4342.95,5
PLN025,Apex Insurance Trust,SELF_FUNDED,MD,4,3244.27,4
PLN030,Frontier Agriculture Co,SELF_FUNDED,SC,3,3788.41,3
PLN011,Lambda Healthcare Partners,FULLY_INSURED,PA,3,2166.29,3
PLN022,Chi Automotive Parts,FULLY_INSURED,IL,3,2452.87,3
PLN035,Keystone Mining Group,SELF_FUNDED,IA,2,1216.73,2


## 5. State Law Compliance Check

Multiple states have passed legislation banning or restricting accumulator
adjustment programs for fully insured health plans. Self-funded (ERISA)
plans are generally exempt from state insurance regulation under federal
preemption.

The check here is straightforward: if a plan is FULLY_INSURED and operates
in a state that bans accumulator programs, it should not be using an
ACCUMULATOR or MAXIMIZER program type. A plan that does is a compliance
risk for the PBM and the carrier.

Note: the list of ban states changes regularly as new legislation passes.
In production this would be a reference table maintained by the compliance
team. For this analysis we use a representative set.

In [0]:
# -- State Accumulator Ban Reference Table --
# In production, this table would be maintained by the compliance team and loaded
# from a managed Delta table (e.g., `compliance.accumulator_ban_states`).
# Each row carries the state code, the effective date of the ban, whether
# the ban covers ACCUMULATOR programs, MAXIMIZER programs, or both, and
# whether ERISA (self-funded) plans are explicitly exempted or if the
# exemption is implicit via federal preemption.
#
# This structure lets the audit handle:
#   - States that ban accumulators but not maximizers (or vice versa)
#   - Bans that took effect mid-year (only claims after effective_date are violations)
#   - Future legislation changes without code modifications

ban_states_data = [
    # (state, effective_date, bans_accumulator, bans_maximizer, erisa_exempt)
    ("AZ", "2020-01-01", True, True, True),   ("AR", "2021-01-01", True, True, True),
    ("CO", "2020-01-01", True, True, True),   ("CT", "2022-01-01", True, True, True),
    ("DE", "2022-01-01", True, True, True),   ("GA", "2020-01-01", True, False, True),
    ("IL", "2022-01-01", True, True, True),   ("IN", "2023-01-01", True, True, True),
    ("KY", "2022-01-01", True, True, True),   ("LA", "2020-01-01", True, True, True),
    ("ME", "2022-01-01", True, True, True),   ("MD", "2022-01-01", True, True, True),
    ("MN", "2023-01-01", True, True, True),   ("MO", "2023-01-01", True, True, True),
    ("MT", "2023-01-01", True, True, True),   ("NV", "2023-01-01", True, True, True),
    ("NH", "2023-01-01", True, True, True),   ("NJ", "2023-01-01", True, True, True),
    ("NM", "2023-01-01", True, True, True),   ("NY", "2020-01-01", True, True, True),
    ("NC", "2023-01-01", True, True, True),   ("ND", "2023-01-01", True, True, True),
    ("OH", "2023-01-01", True, True, True),   ("OK", "2022-01-01", True, True, True),
    ("OR", "2022-01-01", True, True, True),   ("PA", "2024-01-01", True, True, True),
    ("PR", "2022-01-01", True, True, True),   ("SC", "2023-01-01", True, True, True),
    ("SD", "2023-01-01", True, True, True),   ("TN", "2023-01-01", True, True, True),
    ("TX", "2023-01-01", True, True, True),   ("UT", "2022-01-01", True, True, True),
    ("VA", "2020-01-01", True, True, True),   ("WA", "2020-01-01", True, True, True),
    ("WV", "2022-01-01", True, True, True),
]

df_ban_states = spark.createDataFrame(
    ban_states_data,
    ["state_code", "effective_date", "bans_accumulator", "bans_maximizer", "erisa_exempt"]
).withColumn("effective_date", F.to_date("effective_date"))

# Write to Delta for downstream notebooks and dashboard queries
df_ban_states.write.format("delta").mode("overwrite").saveAsTable(
    "pbm_sim.accumulator_ban_states"
)
print(f"State ban reference table written: {df_ban_states.count()} states")
display(df_ban_states.orderBy("state_code"))

State ban reference table written: 35 states


state_code,effective_date,bans_accumulator,bans_maximizer,erisa_exempt
AR,2021-01-01,true,true,true
AZ,2020-01-01,true,true,true
CO,2020-01-01,true,true,true
CT,2022-01-01,true,true,true
DE,2022-01-01,true,true,true
GA,2020-01-01,true,false,true
IL,2022-01-01,true,true,true
IN,2023-01-01,true,true,true
KY,2022-01-01,true,true,true
LA,2020-01-01,true,true,true


In [0]:
# Join plan configurations against the ban reference table.
# The join is program-type-aware: a state that bans ACCUMULATOR but not
# MAXIMIZER won't flag a MAXIMIZER-only plan (e.g., Georgia).
df_compliance = (
    df_plan
    .filter(F.col("funding_model") == "FULLY_INSURED")
    .filter(F.col("accumulator_program_type").isin("ACCUMULATOR", "MAXIMIZER"))
    .join(
        df_ban_states,
        (df_plan["state_domicile"] == df_ban_states["state_code"]) &
        (
            ((df_plan["accumulator_program_type"] == "ACCUMULATOR") & (df_ban_states["bans_accumulator"] == True)) |
            ((df_plan["accumulator_program_type"] == "MAXIMIZER") & (df_ban_states["bans_maximizer"] == True))
        ),
        how="inner"
    )
)

violations = df_compliance.count()

print(f"Plan configuration compliance violations: {violations}")

if violations > 0:
    print("\nPlans using accumulator/maximizer programs in ban states (fully insured):")
    display(
        df_compliance
        .select(
            "plan_id", "plan_name", "funding_model",
            "state_domicile", "accumulator_program_type",
            "effective_date"
        )
    )

log_entry(NB, "state_compliance", "WARNING" if violations > 0 else "SUCCESS",
          row_count=violations,
          detail=f"{violations} fully insured plan(s) using accumulator/maximizer "
                 f"in ban states")

Plan configuration compliance violations: 6

Plans using accumulator/maximizer programs in ban states (fully insured):


plan_id,plan_name,funding_model,state_domicile,accumulator_program_type,effective_date
PLN011,Lambda Healthcare Partners,FULLY_INSURED,PA,ACCUMULATOR,2024-01-01
PLN019,Tau Government Services,FULLY_INSURED,IN,ACCUMULATOR,2023-01-01
PLN029,Evergreen School District,FULLY_INSURED,TN,ACCUMULATOR,2023-01-01
PLN005,Epsilon Teachers Fund,FULLY_INSURED,AZ,ACCUMULATOR,2020-01-01
PLN022,Chi Automotive Parts,FULLY_INSURED,IL,ACCUMULATOR,2022-01-01
PLN036,Lighthouse Credit Union,FULLY_INSURED,CO,ACCUMULATOR,2020-01-01


[PIPELINE_LOG] {"run_id": "RUN_20261017_192721", "timestamp": "2026-10-17T19:28:00.993390+00:00", "notebook": "NB_04", "section": "state_compliance", "status": "WARNING", "row_count": 6, "detail": "6 fully insured plan(s) using accumulator/maximizer in ban states", "duration_sec": 0.0}


In [0]:
# For each violating plan, quantify how many members are affected
# and how many claims involved copay cards. This is the number the
# compliance team needs to scope the remediation.
if violations > 0:
    # Use broadcast join instead of .collect() + .isin() for scalability
    df_violating_plans = F.broadcast(df_compliance.select("plan_id").distinct())

    df_violation_impact = (
        df_claims
        .join(df_violating_plans, on="plan_id", how="inner")
        .agg(
            F.countDistinct("member_id").alias("total_members"),
            F.count("*").alias("total_claims"),
            F.sum(
                F.when(F.col("manufacturer_copay_amount") > 0, 1).otherwise(0)
            ).alias("claims_with_copay_card"),
            F.round(
                F.sum(
                    F.when(F.col("manufacturer_copay_amount") > 0, F.col("manufacturer_copay_amount"))
                     .otherwise(0)
                ), 2
            ).alias("total_mfr_copay_dollars"),
        )
    )

    print("Impact of compliance-violating plans:")
    display(df_violation_impact)

    print("\nEvery member on these plans who used a copay card has potentially been")
    print("subjected to an accumulator program that state law does not allow for")
    print("fully insured coverage. The remediation path depends on whether the")
    print("plan was set up incorrectly (configuration error) or whether the")
    print("plan sponsor intentionally chose this structure without understanding")
    print("the state law constraint.")

Impact of compliance-violating plans:


total_members,total_claims,claims_with_copay_card,total_mfr_copay_dollars
6953,61991,4849,3755300.74



Every member on these plans who used a copay card has potentially been
subjected to an accumulator program that state law does not allow for
fully insured coverage. The remediation path depends on whether the
plan was set up incorrectly (configuration error) or whether the
plan sponsor intentionally chose this structure without understanding
the state law constraint.


## Audit Summary

In [0]:
print(f"{'=' * 60}")
print(f"  COPAY ACCUMULATOR AUDIT RESULTS")
print(f"{'=' * 60}")
print(f"  Claims with manufacturer copay card:   {coupon_count}")
print(f"{'=' * 60}")
print(f"  ACCUMULATOR plan errors (coupon")
print(f"    credited toward deductible):          {error_count} claims")
print(f"  Dollars incorrectly credited:          ${error_dollars:,.2f}")
print(f"{'=' * 60}")
print(f"  MAXIMIZER plan errors:")
print(f"    Pre-exhaustion (mfr $ credited):     {max_pre_error_count} claims")
print(f"    Pre-exhaustion $ incorrectly credited: ${max_pre_error_dollars:,.2f}")
print(f"    Post-exhaustion (member OOP not")
print(f"      credited after card ran out):       {max_post_error_count} claims")
print(f"    Post-exhaustion $ not credited:       ${max_post_error_dollars:,.2f}")
print(f"{'=' * 60}")
print(f"  State compliance violations:           {violations} plan(s)")
if violations > 0:
    for row in df_compliance.collect():
        print(f"    {row['plan_id']} ({row['plan_name']}): "
              f"{row['funding_model']} in {row['state_domicile']}, "
              f"using {row['accumulator_program_type']}")
print(f"{'=' * 60}")

log_entry(NB, "audit_summary", "SUCCESS",
          detail=f"ACCUM errors: {error_count} (${error_dollars:,.2f}), "
                 f"MAX pre: {max_pre_error_count} (${max_pre_error_dollars:,.2f}), "
                 f"MAX post: {max_post_error_count} (${max_post_error_dollars:,.2f}), "
                 f"State violations: {violations}")

  COPAY ACCUMULATOR AUDIT RESULTS
  Claims with manufacturer copay card:   32163
  ACCUMULATOR plan errors (coupon
    credited toward deductible):          60 claims
  Dollars incorrectly credited:          $51,082.96
  MAXIMIZER plan errors:
    Pre-exhaustion (mfr $ credited):     3848 claims
    Pre-exhaustion $ incorrectly credited: $3,000,498.97
    Post-exhaustion (member OOP not
      credited after card ran out):       0 claims
    Post-exhaustion $ not credited:       $0.00
  State compliance violations:           6 plan(s)
    PLN011 (Lambda Healthcare Partners): FULLY_INSURED in PA, using ACCUMULATOR
    PLN019 (Tau Government Services): FULLY_INSURED in IN, using ACCUMULATOR
    PLN029 (Evergreen School District): FULLY_INSURED in TN, using ACCUMULATOR
    PLN005 (Epsilon Teachers Fund): FULLY_INSURED in AZ, using ACCUMULATOR
    PLN022 (Chi Automotive Parts): FULLY_INSURED in IL, using ACCUMULATOR
    PLN036 (Lighthouse Credit Union): FULLY_INSURED in CO, using ACCUMULATO

## Investigation Notes

**ACCUMULATOR errors** need correction at two levels:

1. The affected members' accumulator balances need to be adjusted
to remove the manufacturer copay amounts that were incorrectly credited.

2. The adjudication logic needs to be reviewed to understand why
the ACCUMULATOR program flag wasn't applied correctly for these claims.
Common causes include a timing gap between when the plan was configured
as ACCUMULATOR and when the adjudication engine picked up the change,
or an edge case in the copay card detection logic where the claim
processor didn't recognize the payment source as manufacturer-funded.

**MAXIMIZER errors** have two distinct remediation paths:

1. Pre-exhaustion errors (manufacturer dollars credited): same fix as
ACCUMULATOR errors. Remove the manufacturer amount from accumulators.

2. Post-exhaustion errors (member OOP not credited): the member paid
out of pocket after their copay card ran out, but the system continued
treating the claim as if the manufacturer was still paying. The member's
own cost-sharing needs to be credited toward their deductible and OOPM
retroactively. This is the more harmful error because the member is
overpaying from their own pocket.

**Copay card exhaustion detection** is inherently imprecise without
direct manufacturer program data. The proxy used here (a transition from
nonzero to zero manufacturer_copay_amount on the same drug) works for
most cases but can miss partial exhaustion (reduced but nonzero payments)
and mid-year copay card renewals.

**The state compliance violation** is a separate, more urgent issue. It
requires coordination between the PBM's compliance team, the plan
sponsor, and the carrier. The plan's program type may need to be
changed to TRADITIONAL, and all claims processed under the ACCUMULATOR
or MAXIMIZER logic for affected members would need to be reprocessed
to credit the manufacturer payments toward their accumulators retroactively.

The **state ban reference table** (`pbm_sim.accumulator_ban_states`) is
written by this notebook and should be maintained by the compliance team
as new legislation passes. Each entry carries an effective date so
historical audits apply the correct rule set for the time period.


In [0]:
log_summary(NB)


  PIPELINE LOG SUMMARY: NB_04 (run: RUN_20261017_192721)
  [+] load_source_tables: SUCCESS
  [+] copay_card_baseline: SUCCESS (32163 rows)
  [!] accumulator_errors: WARNING (60 rows)
      60 claims with mfr copay incorrectly credited, $51,082.96
  [!] maximizer_errors: WARNING (3848 rows)
      Pre-exhaustion: 3848 errors ($3,000,498.97), Post-exhaustion: 0 errors ($0.00)
  [!] state_compliance: WARNING (6 rows)
      6 fully insured plan(s) using accumulator/maximizer in ban states
  [+] audit_summary: SUCCESS

